In [145]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm, trange
import nltk
import re

In [146]:
data = pd.read_csv('dump.csv')

In [147]:
data.shape

(83707, 5)

In [148]:
data.head(5)

,Название,Артикул,Категория,Цена без карты,Цена с картой
0,Проходка BORGE Кровельная D6-50мм прямая №1,17162301,"Главная,Каталог,Строительные материалы,Кровля,...",329.33 ₽,312.86 ₽
1,Аэратор ТЕХНОНИКОЛЬ КТВ,17575401,"Главная,Каталог,Строительные материалы,Кровля,...",1 529.00 ₽,1 452.55 ₽
2,Аэратор ТЕХНОНИКОЛЬ Pilot скатный,17575301,"Главная,Каталог,Строительные материалы,Кровля,...",3 008.00 ₽,2 857.60 ₽
3,Проходной элемент ТЕХНОНИКОЛЬ Shinglas,17575701,"Главная,Каталог,Строительные материалы,Кровля,...",1 172.00 ₽,1 113.40 ₽
4,Вентиль ТЕХНОНИКОЛЬ Skat кровельный,17575501,"Главная,Каталог,Строительные материалы,Кровля,...",2 396.00 ₽,2 276.20 ₽


In [149]:
topics = [
          'Главная,Каталог,Строительные материалы,Кровля,Вентиляция для кровли',
          'Главная,Каталог,Строительные материалы,Кровля,Волнистый лист (ондулин)',
          'Главная,Каталог,Строительные материалы,Кровля,Кровельные мастики',
          'Главная,Каталог,Строительные материалы,Кровля,Кровельные ограждения, снегодержатели',
          'Главная,Каталог,Строительные материалы,Кровля,Металлочерепица',
          'Главная,Каталог,Строительные материалы,Кровля,Мягкая черепица',
          'Главная,Каталог,Строительные материалы,Кровля,Профнастил',
          'Главная,Каталог,Строительные материалы,Кровля,Рулонная кровля',
          'Главная,Каталог,Строительные материалы,Строительные сухие смеси и клеи,Затирки',
          'Главная,Каталог,Строительные материалы,Строительные сухие смеси и клеи,Клеи',
          'Главная,Каталог,Строительные материалы,Строительные сухие смеси и клеи,Ровнители для пола',
          'Главная,Каталог,Строительные материалы,Строительные сухие смеси и клеи,Специальные смеси',
          'Главная,Каталог,Строительные материалы,Строительные сухие смеси и клеи,Шпаклевки',
          'Главная,Каталог,Строительные материалы,Строительные сухие смеси и клеи,Штукатурки',
          'Главная,Каталог,Строительные материалы,Строительство из ГКЛ,Гипсокартон',
          'Главная,Каталог,Строительные материалы,Строительство из ГКЛ,Комплектующие для гипсокартона',
          'Главная,Каталог,Строительные материалы,Строительство из ГКЛ,Профиль для гипсокартона',
          'Главная,Каталог,Строительные материалы,Материалы для кладки,Кирпич',
          'Главная,Каталог,Строительные материалы,Материалы для кладки,Пазогребневые блоки и плиты',
          'Главная,Каталог,Строительные материалы,Материалы для кладки,Пенобетонные блоки',
          'Главная,Каталог,Строительные материалы,Металлическая арматура',
          'Главная,Каталог,Строительные материалы,Столярные изделия,Вагонка,  блок-хаус',
          'Главная,Каталог,Строительные материалы,Столярные изделия,Двери жалюзийные',
          'Главная,Каталог,Строительные материалы,Столярные изделия,Деревянные окна',
          'Главная,Каталог,Строительные материалы,Столярные изделия,Доска для пола и комплектующие',
          'Главная,Каталог,Строительные материалы,Столярные изделия,Доска обрезная',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Специальный крепеж',
          'Главная,Каталог,Строительные материалы,Столярные изделия,Лестничные группы',
          'Главная,Каталог,Строительные материалы,Столярные изделия,Погонажные изделия',
          'Главная,Каталог,Строительные материалы,Листовые материалы,OSB',
          'Главная,Каталог,Строительные материалы,Листовые материалы,ДВП',
          'Главная,Каталог,Строительные материалы,Листовые материалы,ДСП',
          'Главная,Каталог,Строительные материалы,Листовые материалы,Металлические листы',
          'Главная,Каталог,Строительные материалы,Листовые материалы,Пластик',
          'Главная,Каталог,Строительные материалы,Листовые материалы,Фанера',
          'Главная,Каталог,Строительные материалы,Материалы для изоляции,Гидроизоляционные материалы',
          'Главная,Каталог,Строительные материалы,Материалы для изоляции,Лента изоляционная и соединительная',
          'Главная,Каталог,Строительные материалы,Материалы для изоляции,Материалы для осушения и защиты от солей кладок старых зданий',
          'Главная,Каталог,Строительные материалы,Материалы для изоляции,Минеральная теплоизоляция',
          'Главная,Каталог,Строительные материалы,Материалы для изоляции,Пенопласт',
          'Главная,Каталог,Строительные материалы,Материалы для изоляции,Пенополистирол',
          'Главная,Каталог,Строительные материалы,Материалы для изоляции,Плёнки и утеплители',
          'Главная,Каталог,Строительные материалы,Материалы для изоляции,Поролон и войлок',
          'Главная,Каталог,Строительные материалы,Материалы для изоляции,Рулонные и битумные кровельные материалы',
          'Главная,Каталог,Строительные материалы,Материалы для изоляции,Уплотнители',
          'Главная,Каталог,Строительные материалы,Материалы для тротуара и ограждение,Бордюры тротуарные и водостоки',
          'Главная,Каталог,Строительные материалы,Материалы для тротуара и ограждение,Водостоки и дренаж',
          'Главная,Каталог,Строительные материалы,Материалы для тротуара и ограждение,Кольца и колодцы',
          'Главная,Каталог,Строительные материалы,Материалы для тротуара и ограждение,Плитка тротуарная и формы',
          'Главная,Каталог,Строительные материалы,Материалы для тротуара и ограждение,Решетки водоприемные',
          'Главная,Каталог,Строительные материалы,Материалы для тротуара и ограждение,Трубы канализационные',
          'Главная,Каталог,Строительные материалы,Материалы для фасадов,Калитки и ограждения',
          'Главная,Каталог,Строительные материалы,Материалы для фасадов,Кованные изделия',
          'Главная,Каталог,Строительные материалы,Материалы для фасадов,Листовая кровля',
          'Главная,Каталог,Строительные материалы,Материалы для фасадов,Сайдинг',
          'Главная,Каталог,Строительные материалы,Материалы для фасадов,Система водоотвода',
          'Главная,Каталог,Двери, окна, замки,Двери межкомнатные',
          'Главная,Каталог,Двери, окна, замки,Двери стальные',
          'Главная,Каталог,Двери, окна, замки,Дверные коробки, наличники и доборы',
          'Главная,Каталог,Двери, окна, замки,Дверные доводчики',
          'Главная,Каталог,Двери, окна, замки,Ручки дверные',
          'Главная,Каталог,Двери, окна, замки,Стопоры дверные',
          'Главная,Каталог,Двери, окна, замки,Петли',
          'Главная,Каталог,Двери, окна, замки,Капители и арки',
          'Главная,Каталог,Двери, окна, замки,Аксессуары для дверей',
          'Главная,Каталог,Двери, окна, замки,Оконная группа,Монтаж окон',
          'Главная,Каталог,Двери, окна, замки,Оконная группа,Москитные сетки и плёнки для окон',
          'Главная,Каталог,Двери, окна, замки,Оконная группа,Оконная фурнитура',
          'Главная,Каталог,Двери, окна, замки,Оконная группа,Откосы',
          'Главная,Каталог,Двери, окна, замки,Оконная группа,Отливы',
          'Главная,Каталог,Двери, окна, замки,Оконная группа,Пластиковые окна',
          'Главная,Каталог,Двери, окна, замки,Оконная группа,Подоконники',
          'Главная,Каталог,Двери, окна, замки,Оконная группа,Уплотнители',
          'Главная,Каталог,Двери, окна, замки,Замки и комплектующие',
          'Главная,Каталог,Двери, окна, замки,Крючки, шпингалеты и задвижки',
          'Главная,Каталог,Двери, окна, замки,Заготовки ключей и брелоки',
          'Главная,Каталог,Двери, окна, замки,Дверцы для кошек',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Анкера',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Верёвки, шнуры',
          'Главная,Каталог,Инструменты и крепёж,Подъём и транспортировка грузов,Комплектующие для транспортировки грузов',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Гвозди',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Дюбели',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Заклёпки',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Крепления для зеркал',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Метрический крепеж, шайбы',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Навесные замки и комплектующие',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Пластины крепежные',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Саморезы',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Такелажный крепёж',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Уголки',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Хомуты, скобы',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Цепи и тросы',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Шурупы',
          'Главная,Каталог,Инструменты и крепёж,Крепёж и скобяные изделия,Шурупы кровельные',
          'Главная,Каталог,Инструменты и крепёж,Электроинструмент,Аккумуляторные дрели и шуруповерты',
          'Главная,Каталог,Инструменты и крепёж,Электроинструмент,Дрели',
          'Главная,Каталог,Инструменты и крепёж,Электроинструмент,Зарядные и пусковые устройства',
          'Главная,Каталог,Инструменты и крепёж,Электроинструмент,Лобзики',
          'Главная,Каталог,Инструменты и крепёж,Электроинструмент,Многофункциональный инструмент',
          'Главная,Каталог,Инструменты и крепёж,Электроинструмент,Перфораторы',
          'Главная,Каталог,Инструменты и крепёж,Электроинструмент,Пилы дисковые, ленточный, сабельные',
          'Главная,Каталог,Инструменты и крепёж,Электроинструмент,Рубанки и рейсмусы',
          'Главная,Каталог,Инструменты и крепёж,Электроинструмент,Углошлифмашины и полировальные машины',
          'Главная,Каталог,Инструменты и крепёж,Электроинструмент,Шлифмашины и граверы',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Биты, насадки',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Буры',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Диски, круги',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Зубила, штробники',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Кордщётки',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Коронки',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Круги полировочные и шлифовальные',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Листы, сетки шлифовальные',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Масла и смазки для инструмента',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Пилки, полотна, ножи',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Расходные материалы для бензопил',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Расходные материалы для компрессоров',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Расходные материалы для минимоек',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Расходные материалы для пылесосов',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Расходные материалы для сварки',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Расходные материалы для степлеров',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Свёрла, патроны',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Фрезы',
          'Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Шарошки',
          'Главная,Каталог,Инструменты и крепёж,Столярно-слесарный инструмент,Гвоздодёры, ломы',
          'Главная,Каталог,Инструменты и крепёж,Столярно-слесарный инструмент,Заклёпочники',
          'Главная,Каталог,Инструменты и крепёж,Столярно-слесарный инструмент,Зубила, кернеры, штробники',
          'Главная,Каталог,Инструменты и крепёж,Столярно-слесарный инструмент,Ключи разводные'
          ]
pack_in_cat = 300

In [150]:
df_res = pd.DataFrame()

for topic in tqdm(topics):
    df_topic = data[data['Категория'] == topic][:pack_in_cat]
    df_res = df_res.append(df_topic, ignore_index=True)

  0%|          | 0/127 [00:00<?, ?it/s]

In [151]:
df_res.shape

(8663, 5)

In [261]:
df_res.head()

,Название,Артикул,Категория,Цена без карты,Цена с картой
0,Проходка BORGE Кровельная D6-50мм прямая №1,17162301,"Главная,Каталог,Строительные материалы,Кровля,...",329.33 ₽,312.86 ₽
1,Аэратор ТЕХНОНИКОЛЬ КТВ,17575401,"Главная,Каталог,Строительные материалы,Кровля,...",1 529.00 ₽,1 452.55 ₽
2,Аэратор ТЕХНОНИКОЛЬ Pilot скатный,17575301,"Главная,Каталог,Строительные материалы,Кровля,...",3 008.00 ₽,2 857.60 ₽
3,Проходной элемент ТЕХНОНИКОЛЬ Shinglas,17575701,"Главная,Каталог,Строительные материалы,Кровля,...",1 172.00 ₽,1 113.40 ₽
4,Вентиль ТЕХНОНИКОЛЬ Skat кровельный,17575501,"Главная,Каталог,Строительные материалы,Кровля,...",2 396.00 ₽,2 276.20 ₽


In [152]:
names = df_res['Название']
type(names)

pandas.core.series.Series

In [263]:
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer("russian")

def token_and_stem(names):
    tokens = [word for sent in nltk.sent_tokenize(names) for word in nltk.word_tokenize(sent)]
    filtered_tokens = []
    for token in tokens:
        if re.search('[а-яА-Я]', token):
            filtered_tokens.append(token)
    stems = [stemmer.stem(t) for t in filtered_tokens]
    return stems

In [264]:
stopwords = nltk.corpus.stopwords.words('russian')
#можно расширить список стоп-слов
stopwords.extend(['что', 'это', 'так', 'вот', 'быть', 'как', 'в', 'к', 'на', 'Главная', 'Каталог'])

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

n_featur=200000
tfidf_vectorizer = TfidfVectorizer(max_df=0.8, max_features=10000,
                                 min_df=0.01, stop_words=stopwords,
                                 use_idf=True, tokenizer=token_and_stem, ngram_range=(1,3))

In [265]:
%%time
tfidf_matrix = tfidf_vectorizer.fit_transform(names)

/usr/local/lib/python3.8/dist-packages/sklearn/feature_extraction/text.py:383: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['бол', 'больш', 'будт', 'быт', 'вед', 'впроч', 'всег', 'всегд', 'главн', 'даж', 'друг', 'е', 'ег', 'ем', 'есл', 'ест', 'ещ', 'зач', 'зде', 'ил', 'иногд', 'каталог', 'когд', 'конечн', 'куд', 'лучш', 'межд', 'мен', 'мног', 'мо', 'можн', 'нег', 'нельз', 'нибуд', 'никогд', 'нич', 'опя', 'посл', 'пот', 'почт', 'разв', 'сво', 'себ', 'совс', 'теб', 'тепер', 'тог', 'тогд', 'тож', 'тольк', 'хорош', 'хот', 'чег', 'чут', 'эт'] not in stop_words.
  warnings.warn('Your stop_words may be inconsistent with '


CPU times: user 2.6 s, sys: 3.99 ms, total: 2.6 s
Wall time: 2.6 s


In [266]:
type(tfidf_matrix)
tfidf_matrix

<8663x98 sparse matrix of type '<class 'numpy.float64'>'
	with 19625 stored elements in Compressed Sparse Row format>

In [267]:
print(tfidf_matrix.shape)

(8663, 98)


In [268]:
tfidf_matrix[:10]

<10x98 sparse matrix of type '<class 'numpy.float64'>'
	with 13 stored elements in Compressed Sparse Row format>

In [269]:
num_clusters = 127

# Метод к-средних - KMeans
from sklearn.cluster import KMeans
km = KMeans(n_clusters=num_clusters)

In [270]:
%%time
km.fit(tfidf_matrix)

CPU times: user 16.5 s, sys: 31.3 ms, total: 16.5 s
Wall time: 2.37 s


KMeans(n_clusters=127)

In [271]:
%%time
idx = km.fit(tfidf_matrix)
clusters = km.labels_.tolist()

#print(clusters)
#print (km.labels_)

CPU times: user 20.3 s, sys: 35.7 ms, total: 20.3 s
Wall time: 2.72 s


In [272]:
len(km.labels_)
clusters[:10]

[1, 29, 29, 29, 29, 29, 29, 29, 29, 29]

In [273]:
clusterkm = km.labels_.tolist()
frame = pd.DataFrame(texts)

#k-means
out = { 'names': names, 'cluster': clusterkm, 'topic': df_res['Категория'] }
frame1 = pd.DataFrame(out, columns = ['names', 'cluster', 'topic'])

In [274]:
frame1.head()

,names,cluster,topic
0,Проходка BORGE Кровельная D6-50мм прямая №1,1,"Главная,Каталог,Строительные материалы,Кровля,..."
1,Аэратор ТЕХНОНИКОЛЬ КТВ,29,"Главная,Каталог,Строительные материалы,Кровля,..."
2,Аэратор ТЕХНОНИКОЛЬ Pilot скатный,29,"Главная,Каталог,Строительные материалы,Кровля,..."
3,Проходной элемент ТЕХНОНИКОЛЬ Shinglas,29,"Главная,Каталог,Строительные материалы,Кровля,..."
4,Вентиль ТЕХНОНИКОЛЬ Skat кровельный,29,"Главная,Каталог,Строительные материалы,Кровля,..."


In [275]:
fp = frame1[frame1.topic.eq('') & frame1.cluster.eq(33)]
print(len(fp))

0


In [276]:
# MiniBatchKMeans
from sklearn.cluster import MiniBatchKMeans

mbk  = MiniBatchKMeans(init='random', n_clusters=num_clusters) #(init='k-means++', ‘random’ or an ndarray)
mbk.fit_transform(tfidf_matrix)
%time mbk.fit(tfidf_matrix)
miniclusters = mbk.labels_.tolist()
print (mbk.labels_)

CPU times: user 363 ms, sys: 3.99 ms, total: 367 ms
Wall time: 183 ms
[28  5  5 ... 39 39 39]


In [277]:
print(topic)
print(num_clusters)

Главная,Каталог,Инструменты и крепёж,Столярно-слесарный инструмент,Ключи разводные
127


In [278]:
for topic in topics:
    print(topic, end = ' \t\t ')
    for cluster in range(num_clusters):
        print(cluster, ' : ', len(frame1[ frame1.topic.eq(topic) &  frame1.cluster.eq(cluster) ]), end = ' ')
    print()

Главная,Каталог,Строительные материалы,Кровля,Вентиляция для кровли 		 0  :  0 1  :  9 2  :  0 3  :  0 4  :  0 5  :  0 6  :  0 7  :  3 8  :  0 9  :  0 10  :  0 11  :  0 12  :  0 13  :  0 14  :  0 15  :  0 16  :  0 17  :  0 18  :  0 19  :  0 20  :  0 21  :  0 22  :  0 23  :  0 24  :  0 25  :  0 26  :  0 27  :  0 28  :  0 29  :  13 30  :  0 31  :  0 32  :  0 33  :  0 34  :  0 35  :  0 36  :  0 37  :  0 38  :  0 39  :  0 40  :  0 41  :  0 42  :  0 43  :  0 44  :  0 45  :  0 46  :  0 47  :  0 48  :  0 49  :  0 50  :  0 51  :  0 52  :  0 53  :  0 54  :  0 55  :  0 56  :  0 57  :  0 58  :  0 59  :  0 60  :  0 61  :  0 62  :  0 63  :  0 64  :  0 65  :  0 66  :  0 67  :  0 68  :  0 69  :  0 70  :  0 71  :  0 72  :  0 73  :  0 74  :  0 75  :  0 76  :  0 77  :  0 78  :  0 79  :  0 80  :  0 81  :  0 82  :  0 83  :  0 84  :  0 85  :  0 86  :  0 87  :  0 88  :  0 89  :  0 90  :  0 91  :  0 92  :  0 93  :  0 94  :  0 95  :  0 96  :  0 97  :  0 98  :  0 99  :  0 100  :  0 101  :  0 102  :  0 103  :  

Главная,Каталог,Строительные материалы,Кровля,Рулонная кровля 		 0  :  0 1  :  0 2  :  0 3  :  0 4  :  0 5  :  0 6  :  0 7  :  0 8  :  0 9  :  0 10  :  0 11  :  0 12  :  0 13  :  0 14  :  0 15  :  0 16  :  0 17  :  0 18  :  0 19  :  0 20  :  0 21  :  0 22  :  0 23  :  0 24  :  0 25  :  0 26  :  0 27  :  0 28  :  0 29  :  0 30  :  0 31  :  0 32  :  0 33  :  0 34  :  0 35  :  0 36  :  0 37  :  0 38  :  0 39  :  0 40  :  0 41  :  0 42  :  0 43  :  0 44  :  0 45  :  0 46  :  0 47  :  0 48  :  0 49  :  0 50  :  0 51  :  0 52  :  0 53  :  0 54  :  0 55  :  0 56  :  0 57  :  0 58  :  0 59  :  0 60  :  0 61  :  0 62  :  0 63  :  0 64  :  0 65  :  0 66  :  0 67  :  0 68  :  1 69  :  0 70  :  0 71  :  0 72  :  0 73  :  0 74  :  0 75  :  0 76  :  0 77  :  0 78  :  0 79  :  0 80  :  0 81  :  0 82  :  0 83  :  0 84  :  0 85  :  0 86  :  0 87  :  0 88  :  0 89  :  0 90  :  0 91  :  0 92  :  0 93  :  0 94  :  0 95  :  0 96  :  0 97  :  0 98  :  0 99  :  0 100  :  0 101  :  0 102  :  0 103  :  0 104  

Главная,Каталог,Строительные материалы,Строительные сухие смеси и клеи,Штукатурки 		 0  :  0 1  :  20 2  :  0 3  :  0 4  :  0 5  :  0 6  :  0 7  :  0 8  :  0 9  :  0 10  :  0 11  :  0 12  :  0 13  :  0 14  :  0 15  :  0 16  :  0 17  :  0 18  :  0 19  :  0 20  :  0 21  :  0 22  :  0 23  :  0 24  :  0 25  :  0 26  :  0 27  :  0 28  :  0 29  :  0 30  :  0 31  :  0 32  :  0 33  :  0 34  :  0 35  :  0 36  :  0 37  :  0 38  :  0 39  :  0 40  :  0 41  :  0 42  :  0 43  :  0 44  :  0 45  :  0 46  :  0 47  :  0 48  :  0 49  :  0 50  :  0 51  :  0 52  :  0 53  :  0 54  :  0 55  :  0 56  :  0 57  :  0 58  :  0 59  :  0 60  :  0 61  :  0 62  :  0 63  :  0 64  :  0 65  :  0 66  :  0 67  :  0 68  :  0 69  :  0 70  :  0 71  :  0 72  :  0 73  :  0 74  :  0 75  :  0 76  :  0 77  :  0 78  :  0 79  :  0 80  :  0 81  :  0 82  :  0 83  :  0 84  :  0 85  :  0 86  :  0 87  :  0 88  :  0 89  :  0 90  :  0 91  :  0 92  :  0 93  :  0 94  :  0 95  :  0 96  :  0 97  :  0 98  :  0 99  :  0 100  :  0 101  :  0 102 

Главная,Каталог,Строительные материалы,Металлическая арматура 		 0  :  0 1  :  33 2  :  0 3  :  0 4  :  0 5  :  0 6  :  2 7  :  74 8  :  0 9  :  0 10  :  0 11  :  0 12  :  0 13  :  0 14  :  0 15  :  0 16  :  0 17  :  0 18  :  0 19  :  0 20  :  0 21  :  0 22  :  0 23  :  0 24  :  0 25  :  0 26  :  0 27  :  0 28  :  0 29  :  0 30  :  0 31  :  0 32  :  0 33  :  0 34  :  0 35  :  0 36  :  0 37  :  0 38  :  0 39  :  0 40  :  0 41  :  0 42  :  0 43  :  0 44  :  0 45  :  0 46  :  0 47  :  0 48  :  0 49  :  0 50  :  0 51  :  0 52  :  0 53  :  0 54  :  0 55  :  5 56  :  0 57  :  0 58  :  0 59  :  0 60  :  0 61  :  0 62  :  0 63  :  7 64  :  0 65  :  0 66  :  12 67  :  0 68  :  0 69  :  0 70  :  0 71  :  0 72  :  0 73  :  0 74  :  0 75  :  0 76  :  0 77  :  0 78  :  0 79  :  0 80  :  0 81  :  0 82  :  0 83  :  0 84  :  0 85  :  0 86  :  0 87  :  0 88  :  0 89  :  0 90  :  0 91  :  0 92  :  0 93  :  0 94  :  0 95  :  0 96  :  0 97  :  0 98  :  0 99  :  0 100  :  0 101  :  0 102  :  0 103  :  0 10

Главная,Каталог,Строительные материалы,Столярные изделия,Лестничные группы 		 0  :  0 1  :  9 2  :  0 3  :  0 4  :  0 5  :  0 6  :  0 7  :  0 8  :  0 9  :  0 10  :  0 11  :  0 12  :  0 13  :  0 14  :  0 15  :  0 16  :  0 17  :  0 18  :  0 19  :  0 20  :  0 21  :  0 22  :  0 23  :  0 24  :  0 25  :  0 26  :  0 27  :  0 28  :  0 29  :  0 30  :  0 31  :  0 32  :  0 33  :  0 34  :  0 35  :  0 36  :  47 37  :  0 38  :  0 39  :  0 40  :  0 41  :  0 42  :  0 43  :  0 44  :  0 45  :  0 46  :  0 47  :  0 48  :  0 49  :  0 50  :  0 51  :  0 52  :  0 53  :  0 54  :  0 55  :  0 56  :  0 57  :  0 58  :  0 59  :  0 60  :  0 61  :  0 62  :  0 63  :  0 64  :  0 65  :  0 66  :  0 67  :  0 68  :  0 69  :  0 70  :  0 71  :  0 72  :  0 73  :  0 74  :  0 75  :  0 76  :  0 77  :  0 78  :  0 79  :  1 80  :  0 81  :  0 82  :  0 83  :  0 84  :  0 85  :  0 86  :  0 87  :  0 88  :  0 89  :  0 90  :  0 91  :  0 92  :  0 93  :  0 94  :  0 95  :  0 96  :  0 97  :  0 98  :  0 99  :  0 100  :  0 101  :  0 102  :  0 1

...

Главная,Каталог,Инструменты и крепёж,Электроинструмент,Перфораторы 		 0  :  0 1  :  28 2  :  0 3  :  0 4  :  0 5  :  0 6  :  0 7  :  0 8  :  0 9  :  0 10  :  0 11  :  0 12  :  0 13  :  0 14  :  0 15  :  0 16  :  0 17  :  0 18  :  0 19  :  0 20  :  0 21  :  0 22  :  0 23  :  0 24  :  0 25  :  0 26  :  0 27  :  0 28  :  0 29  :  0 30  :  0 31  :  0 32  :  0 33  :  0 34  :  0 35  :  0 36  :  0 37  :  0 38  :  0 39  :  0 40  :  4 41  :  0 42  :  0 43  :  0 44  :  0 45  :  0 46  :  0 47  :  0 48  :  0 49  :  0 50  :  0 51  :  0 52  :  0 53  :  0 54  :  0 55  :  0 56  :  0 57  :  0 58  :  0 59  :  0 60  :  0 61  :  0 62  :  0 63  :  0 64  :  0 65  :  0 66  :  0 67  :  0 68  :  0 69  :  0 70  :  0 71  :  0 72  :  0 73  :  0 74  :  0 75  :  0 76  :  0 77  :  0 78  :  0 79  :  0 80  :  0 81  :  0 82  :  0 83  :  0 84  :  0 85  :  0 86  :  0 87  :  0 88  :  0 89  :  0 90  :  0 91  :  0 92  :  0 93  :  0 94  :  0 95  :  0 96  :  0 97  :  0 98  :  0 99  :  0 100  :  0 101  :  0 102  :  0 103  :  0

Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Буры 		 0  :  0 1  :  0 2  :  0 3  :  0 4  :  0 5  :  0 6  :  0 7  :  0 8  :  0 9  :  0 10  :  0 11  :  0 12  :  0 13  :  0 14  :  0 15  :  67 16  :  0 17  :  0 18  :  0 19  :  0 20  :  0 21  :  0 22  :  0 23  :  0 24  :  0 25  :  0 26  :  0 27  :  1 28  :  0 29  :  0 30  :  0 31  :  0 32  :  0 33  :  0 34  :  0 35  :  0 36  :  0 37  :  0 38  :  0 39  :  0 40  :  0 41  :  0 42  :  0 43  :  0 44  :  0 45  :  0 46  :  0 47  :  0 48  :  0 49  :  0 50  :  0 51  :  0 52  :  0 53  :  51 54  :  0 55  :  0 56  :  0 57  :  0 58  :  0 59  :  0 60  :  0 61  :  26 62  :  0 63  :  0 64  :  0 65  :  0 66  :  0 67  :  0 68  :  0 69  :  0 70  :  0 71  :  0 72  :  0 73  :  0 74  :  0 75  :  0 76  :  0 77  :  0 78  :  0 79  :  0 80  :  0 81  :  0 82  :  0 83  :  0 84  :  0 85  :  0 86  :  0 87  :  0 88  :  0 89  :  0 90  :  0 91  :  0 92  :  0 93  :  0 94  :  0 95  :  0 96  :  0 97  :  0 98  :  0 99  :  0 100  :  0 101  :  0 102  :

Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Листы, сетки шлифовальные 		 0  :  0 1  :  21 2  :  0 3  :  0 4  :  0 5  :  0 6  :  0 7  :  0 8  :  0 9  :  0 10  :  21 11  :  0 12  :  0 13  :  0 14  :  0 15  :  0 16  :  0 17  :  47 18  :  0 19  :  0 20  :  0 21  :  0 22  :  0 23  :  0 24  :  0 25  :  0 26  :  0 27  :  0 28  :  1 29  :  0 30  :  0 31  :  0 32  :  0 33  :  0 34  :  0 35  :  0 36  :  0 37  :  0 38  :  0 39  :  0 40  :  0 41  :  0 42  :  0 43  :  0 44  :  0 45  :  0 46  :  0 47  :  0 48  :  0 49  :  23 50  :  0 51  :  0 52  :  0 53  :  0 54  :  0 55  :  0 56  :  0 57  :  26 58  :  0 59  :  0 60  :  0 61  :  0 62  :  0 63  :  0 64  :  1 65  :  0 66  :  1 67  :  4 68  :  0 69  :  0 70  :  0 71  :  0 72  :  0 73  :  0 74  :  0 75  :  0 76  :  0 77  :  0 78  :  0 79  :  0 80  :  0 81  :  0 82  :  0 83  :  0 84  :  0 85  :  0 86  :  0 87  :  0 88  :  0 89  :  0 90  :  0 91  :  0 92  :  0 93  :  0 94  :  0 95  :  0 96  :  0 97  :  0 98  :  0 99  :  0 100

Главная,Каталог,Инструменты и крепёж,Расходные материалы для инструмента,Расходные материалы для пылесосов 		 0  :  0 1  :  49 2  :  0 3  :  0 4  :  0 5  :  0 6  :  0 7  :  0 8  :  0 9  :  0 10  :  0 11  :  0 12  :  0 13  :  0 14  :  0 15  :  0 16  :  0 17  :  0 18  :  0 19  :  0 20  :  0 21  :  0 22  :  0 23  :  0 24  :  0 25  :  0 26  :  0 27  :  0 28  :  0 29  :  0 30  :  0 31  :  0 32  :  0 33  :  0 34  :  0 35  :  0 36  :  0 37  :  0 38  :  0 39  :  0 40  :  0 41  :  0 42  :  0 43  :  0 44  :  0 45  :  0 46  :  0 47  :  0 48  :  0 49  :  0 50  :  0 51  :  0 52  :  0 53  :  0 54  :  0 55  :  0 56  :  0 57  :  0 58  :  0 59  :  0 60  :  0 61  :  0 62  :  0 63  :  0 64  :  0 65  :  0 66  :  0 67  :  0 68  :  0 69  :  0 70  :  0 71  :  0 72  :  0 73  :  0 74  :  0 75  :  0 76  :  0 77  :  0 78  :  0 79  :  0 80  :  0 81  :  0 82  :  0 83  :  0 84  :  0 85  :  0 86  :  0 87  :  0 88  :  0 89  :  0 90  :  0 91  :  0 92  :  0 93  :  0 94  :  0 95  :  0 96  :  0 97  :  0 98  :  0 99  :  0

Главная,Каталог,Инструменты и крепёж,Столярно-слесарный инструмент,Гвоздодёры, ломы 		 0  :  0 1  :  6 2  :  0 3  :  0 4  :  0 5  :  0 6  :  0 7  :  0 8  :  0 9  :  0 10  :  0 11  :  0 12  :  0 13  :  0 14  :  0 15  :  0 16  :  0 17  :  0 18  :  0 19  :  0 20  :  0 21  :  0 22  :  0 23  :  0 24  :  0 25  :  0 26  :  0 27  :  0 28  :  0 29  :  0 30  :  0 31  :  0 32  :  0 33  :  0 34  :  0 35  :  0 36  :  0 37  :  0 38  :  0 39  :  0 40  :  0 41  :  0 42  :  0 43  :  0 44  :  0 45  :  0 46  :  0 47  :  0 48  :  0 49  :  0 50  :  0 51  :  0 52  :  0 53  :  0 54  :  0 55  :  0 56  :  0 57  :  0 58  :  0 59  :  0 60  :  0 61  :  0 62  :  0 63  :  0 64  :  0 65  :  0 66  :  0 67  :  0 68  :  0 69  :  0 70  :  0 71  :  0 72  :  0 73  :  0 74  :  0 75  :  0 76  :  0 77  :  0 78  :  0 79  :  0 80  :  0 81  :  0 82  :  0 83  :  0 84  :  0 85  :  0 86  :  0 87  :  0 88  :  0 89  :  0 90  :  0 91  :  0 92  :  0 93  :  0 94  :  0 95  :  0 96  :  0 97  :  0 98  :  0 99  :  0 100  :  0 101  :  0 102

In [292]:
test_text = 'забор'

In [293]:
text_vec = tfidf_vectorizer.transform([test_text])
pd.DataFrame.sparse.from_spmatrix(text_vec)

,0,1,2,3,4,5,6,7,8,9,...,88,89,90,91,92,93,94,95,96,97
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [294]:
km.predict(text_vec)

array([1], dtype=int32)

In [285]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics.pairwise import cosine_similarity
 
# зададим массив текстов
some_texts = [
   'текст номер один',
   'текст номер два',
   'текст следующий под номером три',
]
df = pd.DataFrame({'texts': some_texts})

In [174]:
find_nearest_to = "документ номер два"

In [175]:
# формирование весов tf-idf
tfidf = TfidfVectorizer()
mx_tf = tfidf.fit_transform(some_texts)
new_entry = tfidf.transform([find_nearest_to])

In [176]:
type(mx_tf)

scipy.sparse.csr.csr_matrix

In [177]:
sdf = pd.DataFrame.sparse.from_spmatrix(mx_tf)
sdf

,0,1,2,3,4,5,6,7
0,0.000000,0.547832,0.000000,0.720333,0.000000,0.000000,0.425441,0.000000
1,0.720333,0.547832,0.000000,0.000000,0.000000,0.000000,0.425441,0.000000
2,0.000000,0.000000,0.479528,0.000000,0.479528,0.479528,0.283217,0.479528


In [178]:
cosine_similarity(mx_tf)

array([[1.        , 0.48111972, 0.12049196],
       [0.48111972, 1.        , 0.12049196],
       [0.12049196, 0.12049196, 1.        ]])

In [179]:
new_entry = tfidf.transform([find_nearest_to])
pd.DataFrame.sparse.from_spmatrix(new_entry)

,0,1,2,3,4,5,6,7
0,0.795961,0.605349,0.0,0.0,0.0,0.0,0.0,0.0


In [180]:
cosine_similarities = cosine_similarity(new_entry, mx_tf).flatten()
cosine_similarities

array([0.33162938, 0.90498638, 0.        ])

In [181]:
# запишем все попарные результаты сравнений
df['cos_similarities'] = cosine_similarities
# и отсортируем по убыванию (т.к. cos(0)=1)
df = df.sort_values(by=['cos_similarities'], ascending=[0])
df

,texts,cos_similarities
1,текст номер два,0.904986
0,текст номер один,0.331629
2,текст следующий под номером три,0.000000
